# ERGT Colab Phase 3 Proof Baseline

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This notebook runs only the proof baseline:

`configs/baseline/proof_wikitext2.json`

It can take a long time on a T4 because the proof config uses `max_steps=10000`, `hidden_dim=512`, `n_layers=6`, and `batch_size=16`.

In [ ]:
# Colab run controls
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

CONFIG_PATH = "configs/baseline/proof_wikitext2.json"
DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
RESULTS_PATH = "runs/phase0_baseline/proof_wikitext2/baseline_results.json"
RUN_DIR = "runs/phase0_baseline/proof_wikitext2"

DEVICE = "cuda"
RUN_BASELINE = True
FORCE_RERUN = False

# Keep False by default. Full archives include large checkpoints.
MAKE_FULL_ARCHIVE = False

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
print("COLAB_TPU_ADDR:", os.environ.get("COLAB_TPU_ADDR"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, set Hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

x = torch.randn(1024, 1024, device="cuda")
y = x @ x
print("cuda_tensor_test:", float(y.mean().detach().cpu()))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True)
print("repo:", repo_head.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"], check=True)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run proof baseline

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

results_path = Path(PROJECT_DIR) / RESULTS_PATH

if RUN_BASELINE:
    if results_path.exists() and not FORCE_RERUN:
        print("Skipping proof baseline because results already exist:", results_path)
    else:
        run_command(
            [
                sys.executable,
                "experiments/train_baseline.py",
                "--config",
                CONFIG_PATH,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_BASELINE is False; no training executed.")

if not results_path.exists():
    raise FileNotFoundError(f"Missing proof baseline result: {results_path}")

## 5. Print proof baseline result

In [ ]:
with results_path.open("r", encoding="utf-8") as handle:
    result = json.load(handle)

print(json.dumps(result, indent=2, sort_keys=True))

required_keys = [
    "final_training_loss",
    "best_validation_loss",
    "final_validation_loss",
    "perplexity",
    "total_training_tokens",
]
missing = [key for key in required_keys if key not in result]
if missing:
    raise RuntimeError(f"Result is missing required keys: {missing}")

## 6. Archive light outputs for review

In [ ]:
light_root = Path("/content/ergt_proof_baseline_light")
if light_root.exists():
    shutil.rmtree(light_root)

source_run_dir = Path(PROJECT_DIR) / RUN_DIR
target_run_dir = light_root / RUN_DIR
target_run_dir.mkdir(parents=True, exist_ok=True)

for path in source_run_dir.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix not in {".json", ".jsonl"}:
        continue
    relative = path.relative_to(source_run_dir)
    destination = target_run_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
    print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive("/content/ergt_proof_baseline_light", "zip", light_root)
print("Light archive ready:", light_archive)
print("Download and share this light zip for review.")

## 7. Optional full archive with checkpoints

In [ ]:
if MAKE_FULL_ARCHIVE:
    full_archive = shutil.make_archive(
        "/content/ergt_proof_baseline_full", "zip", PROJECT_DIR, RUN_DIR
    )
    print("Full archive ready:", full_archive)
else:
    print("Skipping full archive. Set MAKE_FULL_ARCHIVE=True if you need checkpoints.")